# Week 1: ACT on ALOHA Insertion

Settings: Accelerator = GPU T4, Internet = On, Secret `WANDB_API_KEY` attached.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["MUJOCO_GL"] = "egl"
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ.pop("PYTHONPATH", None)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

ALOHA's `labmaze` dependency has no Python 3.13 wheel, so LeRobot goes in a Python 3.12 venv (uv downloads 3.12 if the image lacks it).

In [ ]:
!pip install -q uv
!rm -rf /tmp/lr
!uv venv --python 3.12 /tmp/lr
!uv pip install --python /tmp/lr/bin/python --only-binary labmaze "lerobot[aloha,training]"
!/tmp/lr/bin/python -c "import sys, torch, lerobot, gym_aloha; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())"
assert _exit_code == 0, f"command failed with exit code {_exit_code}"

## Full run
Evals 50 episodes at step 25k and 50k, in-process (`use_async_envs=false`). The step-50k success rate goes into `results.csv`.

In [ ]:
STEPS = 50000
!/tmp/lr/bin/lerobot-train --policy.type=act --dataset.repo_id=lerobot/aloha_sim_insertion_human \
  --env.type=aloha --env.task=AlohaInsertion-v0 \
  --steps={STEPS} --seed=0 --save_freq=25000 --env_eval_freq=25000 \
  --eval.n_episodes=50 --eval.batch_size=10 --eval.use_async_envs=false \
  --wandb.enable=true --wandb.disable_artifact=true --wandb.project=scratch --job_name=act-insertion-{STEPS}-s0 \
  --policy.push_to_hub=false --output_dir=/kaggle/working/act_insertion
assert _exit_code == 0, f"command failed with exit code {_exit_code}"